# 🧪 Sesión 27 · Blindaje OWASP: Prevención Total de SQL Injection (SQLi)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué puede hacer un apóstrofo en tu consulta?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/libros/buscar?titulo=Cien
HTTP/1.1 200 OK
{"libros":[{"id":1,"titulo":"Cien años de soledad"}]}

GET /api/libros/buscar?titulo=%27%20OR%20%271%27%3D%271
HTTP/1.1 200 OK
{"libros":[]}          # no devuelve toda la tabla

GET /api/libros/1
HTTP/1.1 200 OK
{"id":1,"titulo":"Cien años de soledad"}
```

_El payload se trata como texto, no como instrucción. La consulta mantiene su forma._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Reconocer una consulta vulnerable a SQL Injection.
- Escribir consultas parametrizadas.
- Verificar que un payload malicioso no devuelve datos ajenos.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Una búsqueda normal encuentra un libro → HTTP 200
- [ ] El payload de inyección no devuelve toda la tabla → HTTP 200
- [ ] El acceso por id sigue funcionando → HTTP 200
- [ ] La consulta usa parámetros

## 🧩 El reto

Implementa una búsqueda de libros a prueba de SQL Injection y demuéstralo con el payload clásico.

**Requisitos del contrato**

- Siembra «Cien años de soledad» (id 1) y «Rayuela» (id 2) como datos de prueba.
- GET /api/libros/buscar?titulo=... busca por coincidencia.
- La consulta usa parámetros, nunca concatenación de cadenas.
- El payload ' OR '1'='1 no devuelve toda la tabla.
- GET /api/libros/{id} sigue funcionando.

> **Pista conceptual:** En SQLite: `cursor.execute("SELECT ... WHERE titulo LIKE ?", (f"%{texto}%",))`. El marcador `?` separa la estructura de los datos.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion27.md` comparando la consulta vulnerable y la segura.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu búsqueda segura se verá así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'Una búsqueda normal encuentra un libro',
  'metodo': 'GET',
  'ruta': '/api/libros/buscar?titulo=Cien',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]}},
 {'descripcion': 'El payload de inyección no devuelve toda la tabla',
  'metodo': 'GET',
  'ruta': '/api/libros/buscar?titulo=%27%20OR%20%271%27%3D%271',
  'estado': 200,
  'json_contiene': {'libros': []}},
 {'descripcion': 'El acceso por id sigue funcionando',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Cien años de soledad'}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["\\?|:titulo|text\\(|like\\(|where\\(|select\\("], descripcion="La consulta usa parámetros"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion27.md` a `auditorias/auditoria_sesion27.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
